# 🧠 Introducción a Redes Neuronales con Deep Learning
## Reconocimiento de dígitos escritos a mano — Dataset MNIST

---

### ¿Qué vamos a hacer?
Vamos a entrenar una **red neuronal** para que aprenda a reconocer dígitos escritos a mano (0 al 9).  
Es el "Hola Mundo" del Deep Learning.

### Flujo del notebook:
```
1. Importar librerías
2. Cargar y explorar el dataset
3. Preprocesar los datos
4. Construir la red neuronal
5. Entrenar el modelo
6. Evaluar el rendimiento
7. Visualizar resultados
8. Hacer predicciones nuevas
```

> 📌 **TensorFlow y Keras ya vienen instalados en Google Colab. No necesitás instalar nada.**

---
## 1. 📦 Importar librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")
print("✅ Librerías importadas correctamente")

---
## 2. 🔍 Cargar y explorar el dataset MNIST

MNIST contiene **70.000 imágenes** de dígitos escritos a mano:
- 60.000 para entrenamiento
- 10.000 para prueba

Cada imagen es de **28x28 píxeles** en escala de grises.

In [ ]:
# Cargar el dataset (se descarga automáticamente la primera vez)
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print("=== FORMA DE LOS DATOS ===")
print(f"X_train: {X_train.shape}  → {X_train.shape[0]} imágenes de {X_train.shape[1]}x{X_train.shape[2]} píxeles")
print(f"y_train: {y_train.shape}  → etiquetas (0 al 9)")
print(f"X_test:  {X_test.shape}")
print(f"y_test:  {y_test.shape}")

print(f"\nValores de píxel: mín={X_train.min()}, máx={X_train.max()}")
print(f"Clases disponibles: {np.unique(y_train)}")

In [ ]:
# Visualizar algunas imágenes del dataset
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
fig.suptitle('Ejemplos del dataset MNIST', fontsize=14, fontweight='bold')

for i in range(3):
    for j in range(10):
        # Mostrar un ejemplo de cada dígito (0-9)
        idx = np.where(y_train == j)[0][i]
        axes[i, j].imshow(X_train[idx], cmap='gray')
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(f'Dígito {j}', fontsize=9)

plt.tight_layout()
plt.show()

---
## 3. ⚙️ Preprocesar los datos

Las redes neuronales trabajan mejor con valores entre 0 y 1.  
Vamos a **normalizar** dividiendo por 255 (valor máximo de un píxel).

También vamos a **"aplanar"** las imágenes: de 28x28 → 784 valores en línea.

In [ ]:
# Normalizar: pasar de rango [0, 255] a [0, 1]
X_train_norm = X_train / 255.0
X_test_norm  = X_test  / 255.0

print("Antes de normalizar:", X_train.min(), "-", X_train.max())
print("Después de normalizar:", X_train_norm.min(), "-", X_train_norm.max())

# Visualizar cómo "ve" la red un dígito: como vector de 784 números
print(f"\nForma original de una imagen: {X_train[0].shape}")
print(f"La red la recibe como: {X_train_norm[0].flatten().shape}")

---
## 4. 🏗️ Construir la red neuronal

Vamos a construir una red **densa (fully connected)** con esta arquitectura:

```
Entrada (784)  →  Capa oculta 1 (128 neuronas, ReLU)  →  Capa oculta 2 (64 neuronas, ReLU)  →  Salida (10 neuronas, Softmax)
```

- **Flatten**: convierte la imagen 28x28 en un vector de 784
- **Dense**: capa donde cada neurona está conectada con todas las anteriores
- **ReLU**: función de activación que agrega no-linealidad (`max(0, x)`)
- **Softmax**: convierte la salida en probabilidades (suman 1.0)

In [ ]:
# Definir la arquitectura de la red
model = keras.Sequential([
    # Capa de entrada: aplana la imagen 28x28 en un vector de 784
    layers.Flatten(input_shape=(28, 28), name='entrada'),

    # Primera capa oculta: 128 neuronas con activación ReLU
    layers.Dense(128, activation='relu', name='capa_oculta_1'),

    # Dropout: apaga el 20% de neuronas al azar durante el entrenamiento
    # Esto evita el sobreajuste (overfitting)
    layers.Dropout(0.2, name='dropout'),

    # Segunda capa oculta
    layers.Dense(64, activation='relu', name='capa_oculta_2'),

    # Capa de salida: 10 neuronas (una por dígito), Softmax da probabilidades
    layers.Dense(10, activation='softmax', name='salida')
], name='red_neuronal_mnist')

# Resumen de la arquitectura
model.summary()

In [ ]:
# Compilar el modelo: definir cómo aprende
model.compile(
    optimizer='adam',                      # Algoritmo de optimización
    loss='sparse_categorical_crossentropy', # Función de pérdida para clasificación
    metrics=['accuracy']                   # Métrica que queremos ver
)

print("✅ Modelo compilado")
print("\n📌 ¿Qué hace cada parte?")
print("  optimizer='adam'   → ajusta los pesos para minimizar el error")
print("  loss=...crossentropy → mide qué tan equivocado está el modelo")
print("  metrics=['accuracy'] → queremos ver el % de aciertos")

---
## 5. 🚀 Entrenar el modelo

Durante el entrenamiento, la red:
1. Ve una imagen
2. Predice un dígito
3. Compara con la respuesta correcta
4. Ajusta sus pesos para equivocarse menos

Un **epoch** = pasar por todo el dataset de entrenamiento una vez.

In [ ]:
# Entrenar la red neuronal
print("🏋️ Entrenando la red neuronal...\n")

history = model.fit(
    X_train_norm, y_train,
    epochs=10,           # Cantidad de veces que recorre el dataset completo
    batch_size=32,       # Cuántas imágenes procesa antes de ajustar pesos
    validation_split=0.1, # 10% del train para validación en tiempo real
    verbose=1
)

print("\n✅ Entrenamiento terminado!")

---
## 6. 📊 Evaluar el rendimiento

Ahora probamos el modelo con datos que **nunca vio** durante el entrenamiento.

In [ ]:
# Evaluar en el conjunto de prueba
loss, accuracy = model.evaluate(X_test_norm, y_test, verbose=0)

print("=== RESULTADOS EN DATOS DE PRUEBA ===")
print(f"  Pérdida (loss):    {loss:.4f}")
print(f"  Precisión:         {accuracy*100:.2f}%")
print(f"  Errores en 10.000: ~{int((1-accuracy)*10000)} imágenes mal clasificadas")

In [ ]:
# Graficar la evolución del entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Evolución del entrenamiento', fontsize=14, fontweight='bold')

epochs_range = range(1, len(history.history['accuracy']) + 1)

# Gráfico de precisión
ax1.plot(epochs_range, history.history['accuracy'], 'b-o', label='Train')
ax1.plot(epochs_range, history.history['val_accuracy'], 'r-o', label='Validación')
ax1.set_title('Precisión por época')
ax1.set_xlabel('Época')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gráfico de pérdida
ax2.plot(epochs_range, history.history['loss'], 'b-o', label='Train')
ax2.plot(epochs_range, history.history['val_loss'], 'r-o', label='Validación')
ax2.set_title('Pérdida por época')
ax2.set_xlabel('Época')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Si train sube pero validación baja → overfitting (sobreajuste)")
print("   Si ambas suben juntas → el modelo está aprendiendo bien")

---
## 7. 🔎 Visualizar predicciones y errores

Veamos qué imágenes el modelo clasifica bien y en cuáles se equivoca.

In [ ]:
# Hacer predicciones sobre el set de prueba
y_pred_probs = model.predict(X_test_norm, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)  # Elegir la clase con mayor probabilidad

print(f"Ejemplo de predicción:")
print(f"  Probabilidades: {y_pred_probs[0].round(3)}")
print(f"  Predicción:     {y_pred[0]}")
print(f"  Real:           {y_test[0]}")

In [ ]:
# Mostrar 20 predicciones: verde=correcto, rojo=incorrecto
fig, axes = plt.subplots(4, 5, figsize=(12, 10))
fig.suptitle('Predicciones del modelo (verde=correcto, rojo=error)', fontsize=13, fontweight='bold')

indices = np.random.choice(len(X_test), 20, replace=False)

for i, idx in enumerate(indices):
    ax = axes[i // 5, i % 5]
    ax.imshow(X_test[idx], cmap='gray')
    ax.axis('off')

    correcto = y_pred[idx] == y_test[idx]
    color = 'green' if correcto else 'red'
    simbolo = '✓' if correcto else '✗'

    ax.set_title(
        f'Real: {y_test[idx]} | Pred: {y_pred[idx]} {simbolo}',
        color=color, fontsize=9, fontweight='bold'
    )

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Matriz de confusión: qué dígitos confunde entre sí
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=range(10))
disp.plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title('Matriz de Confusión\n(diagonal = predicciones correctas)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Los números fuera de la diagonal son los errores.")
print("   ¿Qué pares de dígitos confunde más el modelo?")

---
## 8. 🎯 Hacer una predicción propia

Elegí cualquier imagen del test set y mirá la confianza del modelo en cada clase.

In [ ]:
# ✏️ Cambiá este índice para probar con diferentes imágenes (0 a 9999)
INDICE = 42

imagen = X_test_norm[INDICE]
etiqueta_real = y_test[INDICE]

# La red espera un batch (lote), así que agregamos una dimensión extra
imagen_batch = np.expand_dims(imagen, axis=0)
probabilidades = model.predict(imagen_batch, verbose=0)[0]
prediccion = np.argmax(probabilidades)

# Visualizar
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Imagen
ax1.imshow(X_test[INDICE], cmap='gray')
ax1.axis('off')
correcto = prediccion == etiqueta_real
color = 'green' if correcto else 'red'
ax1.set_title(
    f'Real: {etiqueta_real}  |  Predicción: {prediccion}  {"✓" if correcto else "✗"}',
    fontsize=13, color=color, fontweight='bold'
)

# Barras de probabilidad
colores = ['red' if i == prediccion else 'steelblue' for i in range(10)]
ax2.bar(range(10), probabilidades * 100, color=colores)
ax2.set_xticks(range(10))
ax2.set_xlabel('Dígito')
ax2.set_ylabel('Confianza (%)')
ax2.set_title('Confianza del modelo por dígito')
ax2.grid(axis='y', alpha=0.3)

for i, prob in enumerate(probabilidades):
    if prob > 0.01:
        ax2.text(i, prob * 100 + 0.5, f'{prob*100:.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## 9. 🧪 Experimentos para seguir aprendiendo

¡Ahora es tu turno de experimentar! Probá hacer estos cambios y observá cómo afectan los resultados:

### Cambios sugeridos:

| Experimento | Qué modificar | Qué observar |
|---|---|---|
| Más neuronas | `Dense(256, ...)` en capa 1 | ¿Mejora la precisión? |
| Más capas | Agregar otra `Dense(32, ...)` | ¿Más lento? ¿Más preciso? |
| Más epochs | `epochs=20` | ¿Llegás a overfit? |
| Sin Dropout | Eliminar la capa Dropout | ¿Aparece overfitting? |
| Otro optimizador | `optimizer='sgd'` | ¿Converge más lento? |

### Conceptos clave aprendidos:

- **Flatten**: convierte matriz → vector  
- **Dense (fully connected)**: cada neurona conectada con todas  
- **ReLU**: función de activación no lineal  
- **Softmax**: convierte salidas en probabilidades  
- **Dropout**: regularización para evitar overfitting  
- **Adam**: optimizador que ajusta la tasa de aprendizaje automáticamente  
- **Epochs / Batch size**: cuánto y cómo entrena la red  
- **Overfitting**: cuando la red memoriza en vez de generalizar  

In [ ]:
# 🧪 ZONA DE EXPERIMENTOS
# Copiá y modificá el modelo de arriba para tus propios experimentos

modelo_experimento = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),

    # 👇 Cambiá la arquitectura aquí
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),

    layers.Dense(10, activation='softmax')
])

modelo_experimento.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

modelo_experimento.summary()

# Entrenar
history2 = modelo_experimento.fit(
    X_train_norm, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

# Evaluar
loss2, acc2 = modelo_experimento.evaluate(X_test_norm, y_test, verbose=0)
print(f"\n🔬 Resultado experimento: {acc2*100:.2f}%")
print(f"   Modelo original:      {accuracy*100:.2f}%")
print(f"   Diferencia:           {(acc2-accuracy)*100:+.2f}%")